# W2 device_name ↔ Physical Label Mapper

这个 notebook 只用于**首次把协议中的 `device_name` 对应到真实物理 W2**，不接入正式上位机。

已经验证当前 5 块设备的 `READ 0x03` 分别返回 `RunE W2 1` ~ `RunE W2 5`。因此这里采用一个更简单的长期规则：

```text
protocol device_name = RunE W2 5
        ↓
physical sticker     = W2-5
        ↓
logical device_id    = w2_5
```

这样不再维护第二套 `Sensor-A ↔ RunE W2 5` registry。首次确认实体之后，直接给设备贴与协议编号一致的标签。

## 标定方式

本工具**不使用尚未验证 payload 的 shutdown(0x14)**。它只使用已经在项目中实机使用过的：

- `READ 0x03`：发现当前设备的 protocol identity；
- `stop_collect()`：先停止所有候选 W2；
- `start_emg_raw()`：一次只让目标 `device_name` 采集。

识别目标时，连续轻触/捏住/晃动候选设备的电极端或导线。只有当前目标 W2 的输入扰动会出现在下面的实时 `RMS / STD / peak-to-peak` 指标中。确认实体后立即贴上推荐标签，例如 `W2-5`。

> `COMx` 只在本次运行中用于打开串口；每次 notebook 启动都会重新通过协议读取 `device_name`，不会保存 COM ↔ device 的长期关系。


In [ ]:
from __future__ import annotations

import math
import re
import sys
import time
from collections import deque
from pathlib import Path

import pandas as pd
import serial
from serial.tools import list_ports
from IPython.display import display

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / 'DeviceInterface' / 'w2_protocol.py').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing DeviceInterface/w2_protocol.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from DeviceInterface.w2_protocol import (
    W2CommandBuilder,
    W2RawPacket,
    W2StreamParser,
    W2_NOTIFY_HEADER,
    W2_NOTIFY_TAIL,
)

BAUD_RATE = 256000
SERIAL_TIMEOUT_S = 0.05
QUERY_WINDOW_S = 0.8
QUERY_IDLE_S = 0.12

print('REPO_ROOT:', REPO_ROOT)
print('pyserial :', serial.VERSION)


## 1. 自动发现当前连接的 W2

这里会遍历当前串口，并实际发送 `READ 0x03`。只有返回合法 `RunE W2 ...` device name 的串口才会进入 `DISCOVERED_W2`。


In [ ]:
def _read_response_bytes(handle: serial.Serial, total_window_s: float = QUERY_WINDOW_S) -> bytes:
    deadline = time.monotonic() + total_window_s
    last_data_at: float | None = None
    chunks: list[bytes] = []
    while time.monotonic() < deadline:
        waiting = int(getattr(handle, 'in_waiting', 0) or 0)
        chunk = bytes(handle.read(waiting if waiting > 0 else 1))
        if chunk:
            chunks.append(chunk)
            last_data_at = time.monotonic()
            continue
        if last_data_at is not None and time.monotonic() - last_data_at >= QUERY_IDLE_S:
            break
    return b''.join(chunks)


def _matching_payload(raw: bytes, frame_type: int) -> bytes | None:
    buffer = bytearray(raw)
    while buffer:
        try:
            header_index = buffer.index(W2_NOTIFY_HEADER)
        except ValueError:
            return None
        if header_index:
            del buffer[:header_index]
        if len(buffer) < 2:
            return None
        frame_len = int(buffer[1]) + 3
        if frame_len < 6 or len(buffer) < frame_len:
            return None
        frame = bytes(buffer[:frame_len])
        del buffer[:frame_len]
        if frame[-1] != W2_NOTIFY_TAIL:
            continue
        if frame[3] != (frame[1] ^ frame[2]):
            continue
        if frame[2] == frame_type:
            return frame[4:-1]
    return None


def read_device_name(port: str) -> str | None:
    try:
        with serial.Serial(
            port, BAUD_RATE, bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE, stopbits=serial.STOPBITS_ONE,
            timeout=SERIAL_TIMEOUT_S,
        ) as handle:
            handle.reset_input_buffer()
            handle.write(W2CommandBuilder.stop_collect())
            handle.flush()
            time.sleep(0.08)
            handle.reset_input_buffer()
            handle.write(W2CommandBuilder.read(W2CommandBuilder.ADDRESS_DEVICE_NAME))
            handle.flush()
            raw = _read_response_bytes(handle)
            payload = _matching_payload(raw, W2CommandBuilder.ADDRESS_DEVICE_NAME)
            if payload is None:
                return None
            return payload.strip(b'\x00').decode('utf-8').strip()
    except (OSError, serial.SerialException, UnicodeDecodeError):
        return None


def device_number(device_name: str) -> int | None:
    match = re.search(r'(\d+)\s*$', device_name)
    return int(match.group(1)) if match else None


def recommended_physical_label(device_name: str) -> str:
    number = device_number(device_name)
    return f'W2-{number}' if number is not None else device_name.replace(' ', '-')


def recommended_logical_id(device_name: str) -> str:
    number = device_number(device_name)
    return f'w2_{number}' if number is not None else re.sub(r'[^a-z0-9]+', '_', device_name.lower()).strip('_')


def discover_w2_devices() -> dict[str, str]:
    discovered: dict[str, str] = {}
    rows: list[dict[str, object]] = []
    for port in list_ports.comports():
        name = read_device_name(port.device)
        if not name or not name.startswith('RunE W2'):
            continue
        if name in discovered:
            raise RuntimeError(f'Duplicate W2 device_name detected: {name!r}')
        discovered[name] = port.device
        rows.append({
            'device_name': name,
            'current_port': port.device,
            'recommended_physical_label': recommended_physical_label(name),
            'recommended_logical_id': recommended_logical_id(name),
            'usb_location_observation': port.location,
        })
    rows.sort(key=lambda row: device_number(str(row['device_name'])) or 10**9)
    display(pd.DataFrame(rows))
    return discovered


DISCOVERED_W2 = discover_w2_devices()
print('Discovered:', DISCOVERED_W2)


## 2. 一次只激活一个 protocol device

`observe_target()` 会先再次通过 `device_name` 找到这次运行中的当前串口，然后启动该设备的 EMG raw。

在观察窗口内：

1. 依次轻触/捏住/晃动一块候选 W2 的电极输入或导线；
2. 看 notebook 持续打印的 `STD / P2P / RMS` 是否出现明显跳变；
3. 一旦确认实体，立即贴上推荐标签，例如 `RunE W2 5 → W2-5`；
4. 再测下一台。

建议从 5 → 1 依次确认只是为了操作顺序清晰，没有协议上的特殊含义。


In [ ]:
def _stats(values: deque[float]) -> tuple[float, float, float]:
    if not values:
        return 0.0, 0.0, 0.0
    xs = tuple(values)
    mean = sum(xs) / len(xs)
    variance = sum((x - mean) ** 2 for x in xs) / len(xs)
    std = math.sqrt(variance)
    p2p = max(xs) - min(xs)
    rms = math.sqrt(sum(x * x for x in xs) / len(xs))
    return std, p2p, rms


def observe_target(device_name: str, duration_s: float = 15.0, report_every_s: float = 0.5) -> dict[str, object]:
    if duration_s <= 0 or report_every_s <= 0:
        raise ValueError('duration_s and report_every_s must be positive')

    # Do not trust an old COM mapping. Rediscover immediately before control.
    current = {}
    for port in list_ports.comports():
        name = read_device_name(port.device)
        if name:
            current[name] = port.device

    port = current.get(device_name)
    if port is None:
        raise RuntimeError(f'{device_name!r} is not currently discoverable')

    physical_label = recommended_physical_label(device_name)
    logical_id = recommended_logical_id(device_name)
    print(f'TARGET: {device_name}  -> recommended sticker {physical_label}  -> logical id {logical_id}')
    print(f'Current transport: {port} (temporary; do not record as identity)')
    print('For the next %.1f s, perturb ONE candidate physical W2 at a time.' % duration_s)

    parser = W2StreamParser()
    window: deque[float] = deque(maxlen=1000)
    total_samples = 0
    max_std = 0.0
    max_p2p = 0.0
    max_rms = 0.0

    with serial.Serial(
        port, BAUD_RATE, bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE, stopbits=serial.STOPBITS_ONE,
        timeout=SERIAL_TIMEOUT_S,
    ) as handle:
        handle.reset_input_buffer()
        handle.write(W2CommandBuilder.stop_collect())
        handle.flush()
        time.sleep(0.08)
        handle.reset_input_buffer()
        handle.write(W2CommandBuilder.start_emg_raw())
        handle.flush()

        start = time.monotonic()
        next_report = start
        try:
            while time.monotonic() - start < duration_s:
                chunk = bytes(handle.read(512))
                if chunk:
                    for packet in parser.feed(chunk):
                        if isinstance(packet, W2RawPacket):
                            window.extend(packet.values)
                            total_samples += len(packet.values)
                now = time.monotonic()
                if now >= next_report:
                    std, p2p, rms = _stats(window)
                    max_std = max(max_std, std)
                    max_p2p = max(max_p2p, p2p)
                    max_rms = max(max_rms, rms)
                    print(
                        f'{now-start:5.1f}s | samples={total_samples:6d} | '
                        f'STD={std:10.2f} | P2P={p2p:10.2f} | RMS={rms:10.2f}',
                        flush=True,
                    )
                    next_report = now + report_every_s
        finally:
            handle.write(W2CommandBuilder.stop_collect())
            handle.flush()

    result = {
        'device_name': device_name,
        'recommended_physical_label': physical_label,
        'recommended_logical_id': logical_id,
        'current_port_observation': port,
        'total_samples': total_samples,
        'max_std': max_std,
        'max_p2p': max_p2p,
        'max_rms': max_rms,
    }
    print('Finished:', result)
    return result


## 3. 实际标定

推荐先从 `RunE W2 5` 开始。每确认一台，就直接在实体上贴与协议一致的标签。

如果 15 秒不够，重复执行同一台即可。


In [ ]:
TARGET_DEVICE_NAME = 'RunE W2 5'
RESULT = observe_target(TARGET_DEVICE_NAME, duration_s=15.0)


确认后建议按下面固定规则贴标：

```text
RunE W2 1 -> physical sticker W2-1 -> logical id w2_1
RunE W2 2 -> physical sticker W2-2 -> logical id w2_2
RunE W2 3 -> physical sticker W2-3 -> logical id w2_3
RunE W2 4 -> physical sticker W2-4 -> logical id w2_4
RunE W2 5 -> physical sticker W2-5 -> logical id w2_5
```

下一步只需要修改 `TARGET_DEVICE_NAME`，依次确认 4、3、2、1。


In [ ]:
LABEL_PLAN = [
    {
        'device_name': name,
        'physical_label': recommended_physical_label(name),
        'logical_device_id': recommended_logical_id(name),
    }
    for name in sorted(
        DISCOVERED_W2,
        key=lambda name: device_number(name) or -1,
        reverse=True,
    )
]
display(pd.DataFrame(LABEL_PLAN))


## 4. 标定完成后的系统语义

以后正式采集不再依赖 COM 编号作为身份：

```text
serial enumeration
      ↓
READ 0x03
      ↓
RunE W2 5
      ↓
logical device_id = w2_5
      ↓
experiment placement = FDI / APB / ...
```

COM 只作为这一轮连接 `RunE W2 5` 时解析到的 transport locator。

物理贴纸完成后，真正需要长期记录的是：

```text
protocol device_name
physical label
logical device_id
```

而 `placement` 属于每次实验/session 的配置，不属于设备永久身份。


## 5. 最简 device_name 输出工具

这个 cell 只做一件事：扫描当前串口，通过 W2 协议 `READ 0x03` 读取并输出当前连接 W2 的 `device_name`。输出中不包含 COM 号。


In [ ]:
def print_connected_w2_device_names() -> list[str]:
    names: list[str] = []
    for port in list_ports.comports():
        name = read_device_name(port.device)
        if name and name.startswith('RunE W2'):
            names.append(name)

    names.sort(key=lambda name: device_number(name) or 10**9)
    for name in names:
        print(name)
    return names


CONNECTED_W2_DEVICE_NAMES = print_connected_w2_device_names()
